- 对于一些要补 RL 基础的同学，也可以边看 RL4LLM 相关的应用，有的放矢地去补；
    - 基础核心概念，reward，return，V(s), Q(s,a), Advantage，importance sampling, on-policy/off-policy ...

### pg => trpo => ppo

$$
L_{\text{CLIP}}(\theta) =  \mathbb{E} \left[ 
    \min \Big( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \Big) 
\right].
$$

- PO：Policy Optimization
    - 优化的对象是 $\pi_\theta(a|s)$
    - 用 Neural Network 来表示，输入是 $s$，输出是 action probability（conditional）
        - DQN 则是用 Neural Network 表示和优化 $Q_\theta(s,a)$ （joint）
- 带着几个问题
    - $\log \pi_\theta$：是怎么来的；
    - likelihood ratio($r_\theta=\frac{\pi_\theta}{\pi_{old}}$) 是如何引入的；
    - 为什么 ppo-clip 中有没有了 $\log$

#### REINFORCE

- RL objective （utility）
    $$
    J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \gamma^t r_t \right]
    $$
- REINFORCE 给出梯度（用回报 $G_t=\sum_{k\geq t}r^{k-t}r_k$，return to go，未来回报）
    $$
    \nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \right]
    $$
    - utility gradient
    - 更高回报的轨迹更高概率的出现，更低回报的轨迹更低概率的出现
    - 为什么会出现 $\log$,
        - $\nabla_\theta \, \mathbb{E}_{x \sim p_\theta}[f(x)]
= \nabla_\theta \sum_x p_\theta(x) f(x)
= \sum_x f(x) \nabla_\theta p_\theta(x)
= \sum_x f(x) p_\theta(x) \nabla_\theta \log p_\theta(x)
= \mathbb{E}_{x \sim p_\theta}\!\left[f(x) \nabla_\theta \log p_\theta(x)\right].$
        - 通过 log-trick，将“对分布求导”改写成“对 log 概率的梯度 × 权重”的可采样形式，从而可以用蒙特卡洛来无偏估计。
            - 既然是期望（概率加权），就能用样本平均无偏估计
        - 而且，$\nabla_\theta \log p_\theta(\tau) = \sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t)$
    
- DRL 里常把“最大化 $J$” 改写为“最小化 loss”（构造一个等梯度的目标（surrogate loss），让深度学习框架“最小化”它即可：）

    $$
    \mathcal L_{PG}(\theta)=-\mathbb E[G_t\log\pi_\theta(a_t|s_t)]
    $$

    - 实际训练时会加 baseline，等价把 $G_t$ 换成优势 $A_t$
- PG 的 loss 是 $-A_t\log\pi_\theta$（或 $-G_t\log\pi_\theta$），优化它的梯度就是经典的 policy gradient。

#### Advantage (baseline) 

- $\mathbb E[\nabla_\theta\log p_\theta(x)]=0$ (score 的期望为0)，所以可以减去任意基线 $b(s)$ 降方差而不引入偏差
    - $\mathbb E [(f-b)\nabla \log p]=\mathbb E [f\nabla \log p]$
    - 简单证明（依然是 log-trick）：$\mathbb E[\nabla_\theta\log p_\theta(x)]=\int p_\theta(x)\nabla_\theta\log p_\theta(x)dx=\int \nabla p_\theta(x)dx=\nabla_\theta\int p(x)dx=\nabla_\theta 1=0$ 
- 引入状态的基线 $b(s_t)$ 来降方差，定义优势
    - $A_t\doteq G_t-b(s_t)$
    - $Q(s,a)-V(s)$
- 那么“最大化目标（换成最小化 loss）”变为：
    - $\mathcal L_{pg-adv}(\theta)=-\mathbb E[\hat A_t\log\pi_\theta(a_t|s_t)]$

#### importance sampling

> 先用旧策略 $\pi_{old}$ 采样一批轨迹，再在它上面做多步更新。
- 定义 ratio

$$
r_t(\theta)\doteq \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}
$$

- 于是得到 CPI/TRPO 的代理目标（无约束版， CPI：Conservative Policy Iteration，保守策略迭代）：

    $$
    L_{cpi}(\theta)=\mathbb E_{\pi_{old}}[r_t(\theta)\hat A_t]=\int \pi_{old}\frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}\hat A_t
    $$

    - $\nabla_\theta r_t=r_t\nabla_\theta\log\pi_\theta(a_t|s_t)$
    - $\theta=\theta_{old}$ 处，$r_t=1$

    $$
  \nabla_\theta L_{\text{CPI}}(\theta) \Big|_{\theta = \theta_{\text{old}}}
= \mathbb{E} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) \, \hat{A}_t \right],
    $$

- 这正是 vanilla PG（advantage 版本）的梯度。
    - 也就是说，$r_t\hat A_t$ 的代理目标与 $\log\pi\cdot \hat A_t$ 在旧策略附近的一阶梯度方向一致。
    - PG 用 $\log\pi$，CPI 用 $r$

#### TRPO 约束与 PPO 的动机

- trpo 在 $L_{cpi}$ 上加 KL trust region
    - $$\text{Surrogate loss:} \quad \max_{\pi} L(\pi) = \mathbb{E}_{\pi_{\text{old}}} \left[ \frac{\pi(a|s)}{\pi_{\text{old}}(a|s)} A^{\pi_{\text{old}}}(s, a) \right]$$
    - $$\text{Constraint:} \quad \mathbb{E}_{\pi_{\text{old}}} [KL(\pi||\pi_{\text{old}})] \le \epsilon$$
    - 过程理解（有约束的最优化问题）：先用 $\pi_{old}$ 采集 batch trajectory，然后在 gradient 的方向上，选一个最佳的 $\pi$（以 $\pi_{old}$ 为中心的一块信任域，trust region，“KL-球”），使得代理目标最大

- ppo-clip（通过 clip 的简单方式，实现 trpo 中的 kl constraint，解决 TRPO 无法高效集成进 Deep Learning 的问题）
    $$
    L_{\text{CLIP}}(\theta) =  \mathbb{E} \left[ 
        \min \Big( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \Big) 
    \right].
    $$
    - $\hat A_t\gt 0$（这个动作是“好”的，应该增大概率）
        - $r_t\gt (1+\epsilon)$，涨得太猛 → 剪到 $1+\epsilon$
        - $r_t\lt 1-\epsilon$（好动作概率太低了），不 clip，梯度自然把 $r_t$ 拉回去
    - $\hat A\lt 0$（这个动作是“坏”的，应该降低概率）
        - $r_t\gt 1+\epsilon$（坏动作概率太高了），不 clip，梯度自然把 $r_t$ 降下去
        - $r_t\lt 1-\epsilon$：降得太猛 → 剪到 $1-\epsilon$
    - 只在“有利方向”超出阈值时截平（防止过度乐观），在不利方向不截平（让惩罚保真）。

### ppo-clip 

- 探索和利用
    - dapo：3.1
    - 高概率 (exploitation) 与低概率 (exploration)
        - PPO 的上限约束确实限制了低概率动作的概率增加，从而抑制了探索能力。
    - 举例
        - $\epsilon=0.2,\hat A_{i,t}\gt 0$，the system tries to increase the probability
            - $\pi_{\theta_{old}}(o_i|q)=0.01, 0.9$
            - $\pi_{\theta_{old}}(o_i|q)(1+\epsilon)=0.012, 1.08$
            - This implies that ‘exploitation’ tokens with a higher probability (e.g., 0.9) are not constrained to get even extremely larger probabilities like 0.999. Conversely, for low-probability ‘exploration’ tokens, achieving a non-trivial increase in probability is considerably more challenging.